# Cleaning other tables

1. Checking in lsoa_info if any lsoa code is missing from official list
2. Create month_info table
3. Operations on lsoa_demographics as written in plan for db

In [1]:
import geopandas as gpd
import sqlite3
import pandas as pd
import holidays
import matplotlib.pyplot as plt
import libpysal
import seaborn as sns

conn = sqlite3.connect('../data/wales_data.db')
cursor = conn.cursor()

In [2]:
#1 check in lsoa_info for any missing LSOAs

#clean data gathered
clean_lsoas_df = pd.read_csv("../data/wimd-2025-index.csv", usecols=['LSOA code', 'LSOA name'])
total_lsoa_set = set(clean_lsoas_df['LSOA code'])

#get from db all lsoas
db_lsoas_df = pd.read_sql("SELECT lsoa_code FROM lsoa_info;",  conn)
db_lsoa_set = set(db_lsoas_df['lsoa_code'])

#set difference to find which lsoa's missing from our db
missing_lsoas = total_lsoa_set - db_lsoa_set

print(f'total lsoa codes count (clean) {len(total_lsoa_set)}')
print(f'database lsoa codes count {len(db_lsoa_set)}')
print(f'number of missing lsoas {len(missing_lsoas)}')

#convert missing lsoas to series
missing_series = pd.Series(list(missing_lsoas))

#take the first letter in each code and count
code_missing = missing_series.str[0].value_counts()

print(f'missing lsoas come from {code_missing}')

total lsoa codes count (clean) 1917
database lsoa codes count 1917
number of missing lsoas 0
missing lsoas come from Series([], Name: count, dtype: int64)


In [3]:
#2 create month_info table

#getting all months
month_df = pd.read_sql("SELECT DISTINCT month FROM street_crimes ORDER BY month;", conn)

#temporary datetime column
dt_col = pd.to_datetime(month_df['month'])

#days in month built-in function by pandas
month_df['days_in_month'] = dt_col.dt.days_in_month

#mapping seasons
def get_season(month_num):
    if month_num in [3, 4, 5]: return 'spring'
    elif month_num in [6, 7, 8]: return 'summer'
    elif month_num in [9, 10, 11]: return 'fall'
    else: return 'winter'

month_df['season'] = dt_col.dt.month.apply(get_season)
print(f'number of months in database {len(month_df)}')

#holiday counts of wales uk
wales_holidays = holidays.UK(subdiv='Wales')

def count_holidays(date_obj):
    #list of all dates per month
    days_in_this_month = pd.date_range(
        start=date_obj.replace(day=1), 
        periods=date_obj.days_in_month, 
        freq='D'
    )
    #count how many of those dates exist in the official holiday dictionary
    return sum(1 for day in days_in_this_month if day in wales_holidays)

month_df['holiday_count'] = dt_col.apply(count_holidays)

#write table back to database
month_df.to_sql("month_info", conn, if_exists="replace", index=False)

# create index for tft
conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_month_info_month ON month_info(month);")

print(month_df.head(12))

number of months in database 183
      month  days_in_month  season  holiday_count
0   2010-12             31  winter              4
1   2011-01             31  winter              2
2   2011-02             28  winter              0
3   2011-03             31  spring              0
4   2011-04             30  spring              3
5   2011-05             31  spring              2
6   2011-06             30  summer              0
7   2011-07             31  summer              0
8   2011-08             31  summer              1
9   2011-09             30    fall              0
10  2011-10             31    fall              0
11  2011-11             30    fall              0


In [4]:
#3 double check nothing is empty in lsoa_demographics
null_query = """
SELECT 
    COUNT(*) - COUNT(lsoa_code) as missing_lsoa_code,
    COUNT(*) - COUNT(income_score) as missing_income,
    COUNT(*) - COUNT(employment_score) as missing_employment,
    COUNT(*) - COUNT(education_score) as missing_education,
    COUNT(*) - COUNT(health_score) as missing_health,
    COUNT(*) - COUNT(housing_score) as missing_housing,
    COUNT(*) - COUNT(services_score) as missing_services,
    COUNT(*) - COUNT(safety_score) as missing_safety,
    COUNT(*) - COUNT(environment_score) as missing_environment,
    COUNT(*) - COUNT(pop) as missing_pop,
    COUNT(*) - COUNT(child_pop) as missing_child,
    COUNT(*) - COUNT(old_pop) as missing_old,
    COUNT(*) - COUNT(working_pop) as missing_working
FROM lsoa_demographics;
"""

total_rows = pd.read_sql("SELECT COUNT(*) as total FROM lsoa_demographics;", conn).iloc[0,0]
null_df = pd.read_sql(null_query, conn)
null_percentages = (null_df / total_rows) * 100
display(null_percentages.round(2).astype(str) + '%')

,missing_lsoa_code,missing_income,missing_employment,missing_education,missing_health,missing_housing,missing_services,missing_safety,missing_environment,missing_pop,missing_child,missing_old,missing_working
0,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%


In [5]:
#3a Perform transformations on lsoa_demographics to match plan

#loading table into df
df = pd.read_sql("SELECT * FROM lsoa_demographics;", conn)

#calculate the scores as written in plan
df['econ_score'] = (df['income_score'] + df['employment_score']) / 2
df['infrastructure_score'] = (df['education_score'] + df['housing_score'] + df['safety_score'] + df['services_score'] + df['environment_score']) / 5

#calculate pop percentages per group
df['percent_working'] = df['working_pop'] / df['pop']
df['percent_child'] = df['child_pop'] / df['pop']
df['percent_old'] = df['old_pop'] / df['pop']

#filter df to only keep the important cols
columns_to_keep = [
    'lsoa_code', 
    'pop', 
    'econ_score', 
    'infrastructure_score', 
    'health_score', 
    'percent_working', 
    'percent_child', 
    'percent_old'
]
df_clean = df[columns_to_keep]

#overwrite old table
df_clean.to_sql("lsoa_demographics", conn, if_exists="replace", index=False)

#re-apply index just in case it's dropped in process
conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_lsoa_code ON lsoa_demographics(lsoa_code);")
display(df_clean.head())


,lsoa_code,pop,econ_score,infrastructure_score,health_score,percent_working,percent_child,percent_old
0,W01000003,2431,30.55,21.24,27.4,0.606335,0.168655,0.313863
1,W01000004,1283,11.85,19.72,4.5,0.531567,0.133281,0.451286
2,W01000005,1599,20.75,28.48,15.1,0.519074,0.114447,0.443402
3,W01000006,1602,9.40,24.70,12.5,0.586142,0.152310,0.357054
4,W01000007,1744,10.90,26.12,2.2,0.543005,0.150803,0.398509


In [6]:
conn.close()